# [3] Naive Trial with RoBERTa

## Imports

In [ ]:
import env

In [ ]:
from epidec.datasets import SWUnivDaconDataset

from transformers import pipeline, AutoTokenizer
from torch.utils.data import DataLoader

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import json
import sys

## Load Datasets

In [ ]:
DATA_ROOT = "./data"

train_dataset = SWUnivDaconDataset(DATA_ROOT, train=True, valid_ratio=0.1)
valid_dataset = SWUnivDaconDataset(DATA_ROOT, valid=True, valid_ratio=0.1)
test_dataset = SWUnivDaconDataset(DATA_ROOT, train=False)

print(f"INFO: Dataset loaded successfully. Train - {len(train_dataset)}, Valid - {len(valid_dataset)}, Test - {len(test_dataset)}")

In [ ]:
train_dataset[1]

In [ ]:
valid_dataset[1]

## Define Model

In [ ]:
model_id = "roberta-large-openai-detector"

pipe = pipeline("text-classification", model=model_id)
tokenizer = AutoTokenizer.from_pretrained(model_id)

In [ ]:
def chunk_text(text, tokenizer=tokenizer, max_length=400):
    words = text.split()
    chunks = []
    current_chunk = []
    current_length = 0

    for word in words:
        word_tokens = len(tokenizer.encode(word, add_special_tokens=False))
        if current_length + word_tokens > max_length and current_chunk:
            chunks.append(" ".join(current_chunk))
            current_chunk = [word]
            current_length = word_tokens
        else:
            current_chunk.append(word)
            current_length += word_tokens

    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

## Evaluation

In [ ]:
check_only_for = None
threshold = 0.7
mini_batch_size = 4

In [ ]:
# Validation
corrects, errors, true_human, false_human, results = [], [], [], [], []
progress = tqdm(DataLoader(valid_dataset, batch_size=1, shuffle=True), desc="Validating...")

for idx, data in enumerate(progress):
    label = data[1][0]
    data = data[0][0]

    if check_only_for is not None and label != check_only_for:
        continue  # Skip if the label does not match the specified check

    chunks = chunk_text(data)
    preds = [threshold, threshold]
    preds_all = []

    for chunk in chunks:
        pred = pipe(chunk)[0]['score']
        preds_all.append(pred)
        if pred < preds[0]:
            preds[0] = pred
        elif pred > preds[1]:
            preds[1] = pred

    if preds[0] >= threshold and preds[1] >= threshold:
        predicted = 1
    elif preds[0] < threshold and preds[1] < threshold:
        predicted = 0
    else:
        predicted = np.mean(preds_all)

    result = dict(question=data, label=label, predicted=predicted)
    predicted_label = 1 if predicted >= threshold else 0
    if predicted_label == label:
        corrects.append(result)
        if label == 0:
            true_human.append(result)
    else:
        errors.append(result)
        print(f"ERROR: Incorrect prediction for index {idx}, expected {label}, got {predicted}")
        if label == 0:
            false_human.append(result)
    results.append(result)
    progress.set_description(f"Correct: {len(corrects)}/{len(results)} [H: {len(true_human)}, A: {len(corrects)-len(true_human)}], Errors: {len(errors)}/{len(results)} [H: {len(false_human)}, A: {len(errors)-len(false_human)}]")

print(f"INFO: Correct: {len(corrects)}/{len(results)}, Errors: {len(errors)}/{len(results)}")

In [ ]:
# Test
results = []
for idx, data in enumerate(tqdm(test_dataset, desc="Testing...")):
    data = data[0]
    chunks = chunk_text(data)
    preds = [threshold, threshold]
    preds_all = []

    for chunk in chunks:
        pred = pipe(chunk)[0]['score']
        preds_all.append(pred)
        if pred < preds[0]:
            preds[0] = pred
        elif pred > preds[1]:
            preds[1] = pred

    if preds[0] >= threshold and preds[1] >= threshold:
        predicted = 1
    elif preds[0] < threshold and preds[1] < threshold:
        predicted = 0
    else:
        predicted = np.mean(preds_all)

    result = dict(question=data, label=predicted)
    results.append(result)

In [ ]:
r = [1 if results['label'] >= threshold else 0 for results in results]

In [ ]:
sub = pd.read_csv("./data/swuniv_dacon/" + test_dataset.submission_file, encoding='utf-8-sig')

In [ ]:
sub

In [ ]:
sub['generated'] = r

In [ ]:
sub

In [ ]:
sub.to_csv("./data/submission_roberta.csv", index=False, encoding='utf-8-sig')